# 03 — BEDOL mechanism ablation study


**Purpose.** Retrain only the variants needed to interrogate the BEDOL mechanism. This notebook is separate because ablations are computationally expensive. It includes fixed transition, no site embedding, no Bayesian latent, no NLinear anchor, no teacher forcing, and an optional independent-site experiment.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
from torch.utils.data import Subset

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT/"src"))
from bedol_repro import *

DATA_DIR, CONFIG_DIR = ROOT/"data", ROOT/"configs"
OUT = ROOT/"reproduction_outputs"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODE = "FAST"  # use PAPER for the three-seed paper-style check
SEEDS = [11] if MODE=="FAST" else [11,29,47]
MAX_EPOCHS = 12 if MODE=="FAST" else 120
PATIENCE = 3 if MODE=="FAST" else 15
RUN_INDEPENDENT_SITE = False

manifest, frames = load_public_data(DATA_DIR, CONFIG_DIR/"site_manifest.csv")
splits = pd.read_csv(CONFIG_DIR/"chronological_splits.csv")
proc, meta, site_to_idx, hist_features, future_features = prepare_frames(frames,splits)
train_ds = SolarWindows(proc,splits,site_to_idx,hist_features,future_features,"train")
val_ds = SolarWindows(proc,splits,site_to_idx,hist_features,future_features,"val")
test_ds = SolarWindows(proc,splits,site_to_idx,hist_features,future_features,"test")


In [ ]:
variants = [
    "fixed_transition",
    "no_site_embedding",
    "no_bayesian_latent",
    "no_nlinear_anchor",
    "no_teacher_forcing",
]
rows=[]
for variant in variants:
    ms=[]
    for seed in SEEDS:
        m,_ = train_one(
            seed, variant, train_ds, val_ds,
            len(hist_features), len(future_features), len(site_to_idx),
            DEVICE, MAX_EPOCHS, PATIENCE, 128
        )
        ms.append(m)
    q = predict_ensemble(ms,test_ds,proc,meta,DEVICE)
    _, macro = summarize_predictions(q, primary_only=True)
    rows.append({"variant":variant, **macro.iloc[0].to_dict()})

ablation = pd.DataFrame(rows)
ablation.to_csv(OUT/"03_ablation_summary.csv",index=False)
display(ablation)


In [ ]:
# Optional independent-site learning.
# Each site receives its own model weights; the architecture is unchanged.
if RUN_INDEPENDENT_SITE:
    independent_predictions=[]
    for site in proc:
        train_idx=[i for i,(s,_) in enumerate(train_ds.items) if s==site]
        val_idx=[i for i,(s,_) in enumerate(val_ds.items) if s==site]
        test_idx=[i for i,(s,_) in enumerate(test_ds.items) if s==site]
        site_models=[]
        for seed in SEEDS:
            m,_=train_one(
                seed,"full",Subset(train_ds,train_idx),Subset(val_ds,val_idx),
                len(hist_features),len(future_features),len(site_to_idx),
                DEVICE,MAX_EPOCHS,PATIENCE,128
            )
            site_models.append(m)
        q=predict_ensemble(site_models,Subset(test_ds,test_idx),proc,meta,DEVICE)
        independent_predictions.append(q)

    independent=pd.concat(independent_predictions,ignore_index=True)
    independent.to_csv(OUT/"03_independent_site_predictions.csv",index=False)
    _, independent_macro=summarize_predictions(independent,primary_only=True)
    display(independent_macro)
